In [ ]:
%matplotlib inline

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

DATA_PATH = 'Pre.csv'   # Put this notebook in the same folder as Pre(1).csv
OUTDIR = 'figures_pre_ln'
os.makedirs(OUTDIR, exist_ok=True)


In [ ]:
# Load table
df = pd.read_csv(DATA_PATH)

# Clean column names
df = df.rename(columns={'Unnamed: 0': 'Compound_ID'})
df['Compound_ID'] = df['Compound_ID'].astype(str)

display(df)


In [ ]:
# Helper functions
def calc_metrics(pred, exp):
    pred = np.asarray(pred, dtype=float)
    exp = np.asarray(exp, dtype=float)
    rmse = np.sqrt(np.mean((pred - exp) ** 2))
    mae = np.mean(np.abs(pred - exp))
    pearson_r, pearson_p = pearsonr(pred, exp)
    spearman_rho, spearman_p = spearmanr(pred, exp)
    return {
        'RMSE': rmse,
        'MAE': mae,
        'Pearson_r': pearson_r,
        'Pearson_p': pearson_p,
        'Spearman_rho': spearman_rho,
        'Spearman_p': spearman_p
    }

metrics_2d = calc_metrics(df['2D Model'], df['Experimental'])
metrics_3d = calc_metrics(df['3D Model'], df['Experimental'])

metrics_df = pd.DataFrame([metrics_2d, metrics_3d], index=['2D Model', '3D Model'])
display(metrics_df.round(4))


In [ ]:
# Figure 1: compound-level comparison (main-text friendly)
plot_df = df.copy()
plot_df = plot_df.sort_values('Experimental', ascending=False).reset_index(drop=True)

x = np.arange(len(plot_df))
fig, ax = plt.subplots(figsize=(8.5, 4.8))

ax.plot(x, plot_df['Experimental'], marker='o', linewidth=2.2, label='Experimental')
ax.plot(x, plot_df['2D Model'], marker='s', linewidth=1.8, label='2D prediction')
ax.plot(x, plot_df['3D Model'], marker='^', linewidth=1.8, label='3D prediction')

ax.set_xticks(x)
ax.set_xticklabels(plot_df['Compound_ID'], rotation=0)
ax.set_xlabel('Compound ID')
ax.set_ylabel('ln(activity)')
ax.set_title('Compound-level comparison of predicted and experimental values')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig1_compound_level_comparison_ln.png'), bbox_inches='tight')
plt.savefig(os.path.join(OUTDIR, 'Fig1_compound_level_comparison_ln.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# Figure 2: parity plots
fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.2), sharex=False, sharey=False)

for ax, col, panel in zip(
    axes,
    ['2D Model', '3D Model'],
    ['A', 'B']
):
    x = df['Experimental'].values
    y = df[col].values

    # parity line
    all_min = min(x.min(), y.min()) - 0.1
    all_max = max(x.max(), y.max()) + 0.1
    ax.plot([all_min, all_max], [all_min, all_max], '--', linewidth=1.5)
    ax.scatter(x, y, s=60, alpha=0.9)

    # annotate compound ID
    for xi, yi, cid in zip(x, y, df['Compound_ID']):
        ax.text(xi + 0.01, yi + 0.01, cid, fontsize=8)

    # metrics
    m = calc_metrics(y, x)
    txt = f"Pearson r = {m['Pearson_r']:.2f}\nRMSE = {m['RMSE']:.2f}\nMAE = {m['MAE']:.2f}"
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.9, edgecolor='0.8'),
            fontsize=9)

    ax.set_xlim(all_min, all_max)
    ax.set_ylim(all_min, all_max)
    ax.set_xlabel('Experimental ln(activity)')
    ax.set_ylabel(f'{col} ln(activity)')
    ax.set_title(f'{panel}. {col} vs Experimental')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig2_parity_ln.png'), bbox_inches='tight')
plt.savefig(os.path.join(OUTDIR, 'Fig2_parity_ln.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# Figure 3: rank-shift plot
rank_df = df[['Compound_ID', '2D Model', '3D Model', 'Experimental']].copy()
rank_df['Rank_2D'] = rank_df['2D Model'].rank(ascending=False, method='min').astype(int)
rank_df['Rank_3D'] = rank_df['3D Model'].rank(ascending=False, method='min').astype(int)
rank_df['Rank_Exp'] = rank_df['Experimental'].rank(ascending=False, method='min').astype(int)

rank_df = rank_df.sort_values('Rank_Exp').reset_index(drop=True)
display(rank_df[['Compound_ID', 'Rank_2D', 'Rank_3D', 'Rank_Exp']])

fig, ax = plt.subplots(figsize=(8.5, 4.8))

xpos = [0, 1, 2]
labels = ['2D rank', '3D rank', 'Experimental rank']

for _, row in rank_df.iterrows():
    y = [row['Rank_2D'], row['Rank_3D'], row['Rank_Exp']]
    ax.plot(xpos, y, marker='o', linewidth=1.8, alpha=0.85)
    ax.text(2.05, row['Rank_Exp'], row['Compound_ID'], va='center', fontsize=9)

ax.set_xticks(xpos)
ax.set_xticklabels(labels)
ax.set_ylabel('Rank (1 = best)')
ax.set_title('Ranking shift from 2D prediction to 3D prediction and experiment')
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig3_rank_shift_ln.png'), bbox_inches='tight')
plt.savefig(os.path.join(OUTDIR, 'Fig3_rank_shift_ln.pdf'), bbox_inches='tight')
plt.show()


In [ ]:
# Figure 4: residual plot
res_df = df[['Compound_ID']].copy()
res_df['Residual_2D'] = df['2D Model'] - df['Experimental']
res_df['Residual_3D'] = df['3D Model'] - df['Experimental']
res_df = res_df.sort_values('Compound_ID').reset_index(drop=True)
display(res_df)

x = np.arange(len(res_df))
width = 0.35

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.axhline(0, linestyle='--', linewidth=1.2, color='0.5')
ax.bar(x - width/2, res_df['Residual_2D'], width=width, label='2D residual')
ax.bar(x + width/2, res_df['Residual_3D'], width=width, label='3D residual')

ax.set_xticks(x)
ax.set_xticklabels(res_df['Compound_ID'])
ax.set_xlabel('Compound ID')
ax.set_ylabel('Prediction - experimental (ln units)')
ax.set_title('Residual comparison on the natural-log scale')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig4_residuals_ln.png'), bbox_inches='tight')
plt.savefig(os.path.join(OUTDIR, 'Fig4_residuals_ln.pdf'), bbox_inches='tight')
plt.show()
